In [1]:
import os
import numpy as np
import pandas as pd

from xgboost import XGBClassifier

MODEL_FILE = "../models_saved/behaviour_aware_xgboost.json"

model = XGBClassifier()

model.load_model(MODEL_FILE)

print("Model loaded successfully.")

Model loaded successfully.


In [2]:
FEATURE_COLUMNS = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "isFlaggedFraud",

    "user_transaction_count_before",
    "previous_transaction_amount",
    "time_since_previous_transaction",
    "previous_average_amount",
    "amount_deviation",
    "amount_to_previous_average",
    "balance_depletion",
    "amount_to_balance_ratio",

    "receiver_transaction_count_before",
    "receiver_previous_amount",
    "receiver_transaction_frequency",

    "user_transfer_count_before",
    "user_cashout_count_before",

    "transaction_velocity",

    "type_CASH_IN",
    "type_CASH_OUT",
    "type_DEBIT",
    "type_PAYMENT",
    "type_TRANSFER"
]

print("Number of features:", len(FEATURE_COLUMNS))

Number of features: 26


In [3]:
def predict_transaction(transaction_data):

    input_df = pd.DataFrame(
        [transaction_data]
    )

    # Make sure all expected features exist
    input_df = input_df.reindex(
        columns=FEATURE_COLUMNS,
        fill_value=0
    )

    # Handle missing values
    input_df = input_df.fillna(0)

    # Generate probability
    probability = model.predict_proba(
        input_df
    )[0, 1]

    return probability

In [4]:
def classify_risk(probability):

    if probability >= 0.5:
        return "FRAUD"

    return "LEGITIMATE"

In [5]:
def risk_level(probability):

    if probability >= 0.8:
        return "HIGH"

    elif probability >= 0.5:
        return "MEDIUM"

    else:
        return "LOW"

In [6]:
TEST_FILE = "../data/processed/test_data.csv"

test_sample = pd.read_csv(
    TEST_FILE,
    nrows=1
)

print(test_sample.T)

                                             0
step                                       632
type                                  TRANSFER
amount                               480404.03
nameOrig                           C1962729026
oldbalanceOrg                        480404.03
newbalanceOrig                             0.0
nameDest                           C1853051693
oldbalanceDest                             0.0
newbalanceDest                             0.0
isFraud                                      1
isFlaggedFraud                               0
user_transaction_count_before                0
previous_transaction_amount                NaN
time_since_previous_transaction            NaN
previous_average_amount                    NaN
amount_deviation                           NaN
amount_to_previous_average                 NaN
balance_depletion                    480404.03
amount_to_balance_ratio               0.999998
receiver_transaction_count_before            0
receiver_prev

In [7]:
test_sample_encoded = pd.get_dummies(
    test_sample,
    columns=["type"],
    dtype=int
)

test_sample_encoded = test_sample_encoded.reindex(
    columns=FEATURE_COLUMNS + ["isFraud"],
    fill_value=0
)

test_sample_encoded = test_sample_encoded.fillna(0)

print(
    "Prepared feature shape:",
    test_sample_encoded[FEATURE_COLUMNS].shape
)

Prepared feature shape: (1, 26)


In [8]:
probability = model.predict_proba(
    test_sample_encoded[FEATURE_COLUMNS]
)[0, 1]

decision = classify_risk(probability)
risk = risk_level(probability)

print("Fraud probability:", probability)
print("Decision:", decision)
print("Risk level:", risk)

print(
    "Actual label:",
    test_sample_encoded["isFraud"].iloc[0]
)

Fraud probability: 0.9999919
Decision: FRAUD
Risk level: HIGH
Actual label: 1
